In [ ]:
# TODO: Once everything is done move functions to a utils file
# TODO: Add description and variables description to all functions

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
ticker = "FBTP"

root = Path("..")
lob_path = root / "data" / "raw" / ticker
fut_path = lob_path / "futures"
ctd_path = lob_path / "ctd"

year = "2022"
month = "08"
day = "01"
date_str = f"{year}_{month}_{day}"

fut_filename = f"futures_lob_freq_1s_{date_str}.parquet"
ctd_filename = f"ctd_lob_freq_1s_{date_str}.parquet"

# Relevant timestamps distribution

In [ ]:
import math

from matplotlib.ticker import FuncFormatter, MultipleLocator


def extract_timestamps_from_lob(lob_path: Path, lob_type: str):
    # TODO: Add description and variables description

    # Validate lob type
    if lob_type not in {"futures", "ctd"}:
        raise ValueError("lob_type must be either 'futures' or 'ctd'")

    # Read lob parquet file
    try:
        lob_df = pd.read_parquet(lob_path, engine="fastparquet")
    except Exception as e:
        print(f"Error reading {lob_path}: {e}")
        return None

    if lob_df.empty:
        print(f"Warning: {lob_path} is empty.")
        return None

    # Drop not relevant columns for futures lob
    if lob_type == "futures":
        columns = ["#RIC", "Domain", "GMT Offset", "Type"]
        lob_df = lob_df.drop(columns=[c for c in lob_df.columns if c in columns], errors="ignore")

    # Check if all rows are NaN and skip if so
    mask = lob_df.notna().any(axis=1)
    if not mask.any():
        print(f"Warning: {lob_path} has rows but all values are NaN.")
        return None

    filename = Path(lob_path).name
    date = lob_df.index[0].date()
    first_timestamp = lob_df.index[0].time()
    last_timestamp = lob_df.index[-1].time()
    last_non_nan_timestamp = lob_df[mask].index[-1].time()

    return {
        "filename": filename,
        "date": date,
        "first timestamp": first_timestamp,
        "last timestamp": last_timestamp,
        "last non nan timestamp": last_non_nan_timestamp,
    }


def collect_timestamps(base_path: Path):
    # TODO: Add description and variables description

    records = []
    lob_type = "futures" if "futures" in str(base_path) else "ctd"

    for year in sorted(os.listdir(base_path)):
        year_path = base_path / year
        if not year_path.is_dir():
            continue
        for month in sorted(os.listdir(year_path)):
            month_path = year_path / month
            if not month_path.is_dir():
                continue
            for filename in sorted(os.listdir(month_path)):
                lob_path = month_path / filename
                record = extract_timestamps_from_lob(lob_path, lob_type)
                if record is not None:
                    records.append(record)

    return pd.DataFrame(records)


def normalize_timestamps(ts: pd.Series):
    # TODO: Add description and variables description

    ts = ts.dropna()
    return np.array([t.hour * 3600 + t.minute * 60 + t.second for t in ts], dtype=np.int32)


def hh_mm_from_seconds(s, pos):
    # TODO: Add description and variables description

    s = int(round(s))
    day_offset = s // (24 * 3600)
    day_seconds = s % (24 * 3600)
    hh = day_seconds // 3600
    mm = (day_seconds % 3600) // 60

    if day_offset == 0:
        return f"{hh:02d}:{mm:02d}"
    return f"D+{day_offset} {hh:02d}:{mm:02d}"


def _pick_step(value, candidates):
    # TODO: Add description and variables description

    return min(candidates, key=lambda c: abs(c - value))


def _round_down_to_step(value: int, step: int) -> int:
    # TODO: Add description and variables description

    return (value // step) * step


def _round_up_to_step(value: int, step: int) -> int:
    # TODO: Add description and variables description

    return ((value + step - 1) // step) * step


def _fmt_tick_scale(step_seconds: int) -> str:
    # TODO: Add description and variables description

    if step_seconds % 3600 == 0:
        h = step_seconds // 3600
        return f"{h}h"
    if step_seconds % 60 == 0:
        m = step_seconds // 60
        return f"{m}m"
    return f"{step_seconds}s"


def align_timestamps_to_session(ts_seconds: np.ndarray, session_start: str = "08:00"):
    # TODO: Add description and variables description

    hh, mm = map(int, session_start.split(":"))
    start_sec = hh * 3600 + mm * 60
    aligned = ts_seconds.astype(np.int32).copy()

    # Shift times after midnight (before session start) to next day axis region
    aligned[aligned < start_sec] += 24 * 3600

    return aligned, start_sec


def suggest_histogram_params(ts_normalized: np.ndarray, target_bins: int = 50):
    # TODO: Add description and variables description

    data_min = int(ts_normalized.min())
    data_max = int(ts_normalized.max())
    span = max(1, data_max - data_min)
    n = len(ts_normalized)

    # Robust bin width (Freedman-Diaconis), fallback to span/target_bins.
    q25, q75 = np.percentile(ts_normalized, [25, 75])
    iqr = float(q75 - q25)
    if n > 1 and iqr > 0:
        fd_width = 2 * iqr / (n ** (1 / 3))
        bin_width = max(1, int(round(fd_width)))
    else:
        bin_width = max(1, int(round(span / target_bins)))

    # Clamp number of bins for readability.
    n_bins = int(math.ceil(span / bin_width))
    if n_bins < 15:
        n_bins = 15
        bin_width = max(1, int(math.ceil(span / n_bins)))
    elif n_bins > 120:
        n_bins = 120
        bin_width = max(1, int(math.ceil(span / n_bins)))

    # Add margin around observed range.
    pad = max(15, int(0.05 * span))
    start = max(0, data_min - pad)
    end = min(2 * 24 * 3600, data_max + pad)

    bins = np.arange(start, end + bin_width, bin_width)
    if len(bins) < 2:
        bins = np.array([start, end + 1])

    # Major ticks (labeled) always aligned to 5-minute multiples
    desired_major = max(300, int((end - start) / 8))
    major_candidates = [
        300,  # 5m
        600,  # 10m
        900,  # 15m
        1800,  # 30m
        3600,  # 1h
        7200,  # 2h
        10800,  # 3h
    ]
    major_tick = _pick_step(desired_major, major_candidates)

    # Minor ticks denser and unlabeled.
    desired_minor = max(100, major_tick // 5)
    minor_candidates = [30, 60, 120, 300, 600, 900, 1800]
    minor_tick = _pick_step(desired_minor, minor_candidates)

    # Slightly wider figure for wider windows.
    if (end - start) <= 30 * 60:
        figsize = (10, 5)
    elif (end - start) <= 2 * 3600:
        figsize = (12, 5)
    else:
        figsize = (14, 5)

    return {
        "start": start,
        "end": end,
        "bins": bins,
        "bin_width": bin_width,
        "major_tick": major_tick,
        "minor_tick": minor_tick,
        "figsize": figsize,
        "span_minutes": span / 60,
        "n_points": n,
    }


# TODO: Allow for better view of highly concentrated data
# TODO: simplify the code
def plot_timestamps(
    ts: pd.Series,
    title: str,
    window=None,
    target_bins: int = 50,
    y_mode: str = "count",
    align_after_midnight: bool = True,
    session_start: str = "08:00",
    save_path: str = None,
    dpi: int = 300,
    minor_ticks_per_major: int = 4,
    show_tick_scale_note: bool = True,
    note_position: str = "upper_left",
):
    # TODO: Add description and variables description

    ts_seconds = normalize_timestamps(ts)
    if len(ts_seconds) == 0:
        print(f"No valid timestamps for: {title}")
        return

    if align_after_midnight:
        ts_plot, _ = align_timestamps_to_session(ts_seconds, session_start=session_start)
    else:
        ts_plot = ts_seconds.copy()

    params = suggest_histogram_params(ts_plot, target_bins=target_bins)

    if window is not None:
        h1, m1 = map(int, window[0].split(":"))
        h2, m2 = map(int, window[1].split(":"))
        start = h1 * 3600 + m1 * 60
        end = h2 * 3600 + m2 * 60

        # If aligned mode and window crosses midnight, move end to next day
        if align_after_midnight and end <= start:
            end += 24 * 3600

        # Keep adaptive bin width but rebuild bins for requested window.
        bw = max(1, params["bin_width"])
        params["start"] = start
        params["end"] = end
        params["bins"] = np.arange(start, end + bw, bw)

    # Filter to plotting window for cleaner normalization in percent mode
    in_window = (ts_plot >= params["start"]) & (ts_plot <= params["end"])
    ts_plot_window = ts_plot[in_window]
    if len(ts_plot_window) == 0:
        print(f"No timestamps in selected window for: {title}")
        return

    fig, ax = plt.subplots(figsize=params["figsize"])

    if y_mode == "percent":
        weights = np.ones(len(ts_plot_window), dtype=float) * (100.0 / len(ts_plot_window))
        ax.hist(
            ts_plot_window,
            bins=params["bins"],
            weights=weights,
            color="steelblue",
            edgecolor="black",
            alpha=0.75,
        )
        ax.set_ylabel("Frequency (%)")
    else:
        ax.hist(
            ts_plot_window,
            bins=params["bins"],
            color="steelblue",
            edgecolor="black",
            alpha=0.75,
        )
        ax.set_ylabel("Frequency (count)")

    minor_ticks_per_major = max(1, int(minor_ticks_per_major))

    ax.set_xlim(params["start"], params["end"])

    # Major ticks with labels
    major_tick = params["major_tick"]
    tick_start = _round_down_to_step(params["start"], major_tick)
    tick_end = _round_up_to_step(params["end"], major_tick)
    major_ticks = np.arange(tick_start, tick_end + 1, major_tick)
    ax.set_xticks(major_ticks)
    ax.xaxis.set_major_formatter(FuncFormatter(hh_mm_from_seconds))
    ax.tick_params(axis="x", which="major", rotation=45, length=6)

    # Minor ticks without labels
    major_interval = major_ticks[1] - major_ticks[0] if len(major_ticks) > 1 else major_tick
    minor_step = major_interval / (minor_ticks_per_major + 1)
    ax.xaxis.set_minor_locator(MultipleLocator(minor_step))
    ax.tick_params(axis="x", which="minor", labelbottom=False, length=3)

    ax.set_xlabel("Time (hh:mm)")
    ax.set_title(
        f"{title}\nspan = {params['span_minutes'] // 60:.0f}:{params ['span_minutes'] % 60:.0f} h, n = {params['n_points']}, bin width = {params['bin_width']}s"
    )
    ax.grid(True, which="major", alpha=0.25, linestyle="-")
    ax.grid(True, which="minor", alpha=0.10, linestyle=":")

    if show_tick_scale_note:
        major_label = _fmt_tick_scale(major_tick)
        minor_step = major_tick / (minor_ticks_per_major + 1)
        if abs(minor_step - round(minor_step)) < 1e-9:
            minor_label = _fmt_tick_scale(int(round(minor_step)))
        else:
            minor_label = f"{minor_step:.1f}s"
        note = f"Major tick: {major_label}\nMinor tick: {minor_label}"
        ax.text(
            0.01,
            0.04,
            note,
            transform=ax.transAxes,
            ha="left",
            va="bottom",
            fontsize=9,
            bbox={
                "boxstyle": "round,pad=0.3",
                "facecolor": "white",
                "alpha": 0.75,
                "edgecolor": "gray",
            },
        )

    plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path, dpi=dpi, bbox_inches="tight")
    plt.show()
    plt.close(fig)


def plot_timestamp_columns(
    df: pd.DataFrame,
    columns,
    target_bins: int = 50,
    window=None,
    y_mode: str = "count",
    align_after_midnight: bool = True,
    session_start: str = "07:55",
    save_dir: Path = None,
    dpi: int = 300,
    file_prefix: str = "",
    minor_ticks_per_major: int = 4,
    show_tick_scale_note: bool = True,
    note_position: str = "upper_left",
    # TODO: Add option to enable/disable title and maybe add custom title prefix
):
    # TODO: Add description and variables description

    for col in columns:
        save_path = None
        if save_dir is not None:
            save_dir.mkdir(parents=True, exist_ok=True)
            col_slug = col.replace(" ", "_")
            save_path = str(save_dir / f"{file_prefix}{col_slug}.png")

        plot_timestamps(
            df[col],
            f"{col.title()} Distribution",
            window=window,
            target_bins=target_bins,
            y_mode=y_mode,
            align_after_midnight=align_after_midnight,
            session_start=session_start,
            save_path=save_path,
            dpi=dpi,
            minor_ticks_per_major=minor_ticks_per_major,
            show_tick_scale_note=show_tick_scale_note,
            note_position=note_position,
        )

In [ ]:
fut_ts_df = collect_timestamps(fut_path)
ctd_ts_df = collect_timestamps(ctd_path)

In [ ]:
plot_timestamp_columns(
    fut_ts_df,
    ["first timestamp", "last timestamp", "last non nan timestamp"],
    y_mode="percent",
)

In [ ]:
plot_timestamp_columns(
    ctd_ts_df,
    ["first timestamp", "last timestamp", "last non nan timestamp"],
    y_mode="percent",
)

# FBTP/FBTS vuoti

In [ ]:
ticker = "FBTS"
data_raw_path = Path("..") / "data" / "raw" / ticker
fut_path = data_raw_path / "futures"
ctd_path = data_raw_path / "ctd"

for year in os.listdir(fut_path):
    for month in os.listdir(fut_path / year):
        for fut_filename in os.listdir(fut_path / year / month):
            df = pd.read_parquet(fut_path / year / month / fut_filename, engine="fastparquet")
            # print(f"Number of rows in {fut_filename}: {len(df)}")